# BulkFormer DX Clinical Methods Comparison (37M)

Runs the clinical RNA-seq dataset on the **37M model** with **all calibration methods** from the [improving_anomalyt_detection_plan](docs/bulkformer-dx/improving_anomalyt_detection_plan.md) and produces comparison figures.

**Methods compared:**

| Method | Description |
|--------|-------------|
| `none` | Empirical cohort-tail + Gaussian z-score (default) |
| `student_t` | Student-t p-values (heavy-tailed; recommended by deep-research) |
| `nb_approx` | TPM-derived NB approximation (ranking support only) |
| `nb_outrider` | OUTRIDER-style NB test in count space |
| `knn_local` | Local cohort calibration (kNN in embedding space) |
| `nll` | TabPFN-style NLL scoring (different score type) |

**Requires:**
- `data/clinical_rnaseq/raw_counts.tsv`, `gene_annotation_v29.tsv`, `sample_annotation.tsv`
- BulkFormer assets and `model/BulkFormer_37M.pt`
- Kernel with `torch`, `torch_geometric`, and CUDA (e.g. `conda activate bulkformer-cuda`) for GPU acceleration

In [7]:
import os
import subprocess
import sys
from pathlib import Path

import torch

ROOT = Path("..").resolve() if (Path("..").resolve() / "bulkformer_dx").exists() else Path(".").resolve()
env = os.environ.copy()
env["PYTHONPATH"] = str(ROOT)

RUNS = ROOT / "runs" / "clinical_methods_37M"
FIGURES = ROOT / "reports" / "figures" / "clinical_methods_comparison"

# GPU detection: use cuda if available, else cpu (or mps on Apple Silicon)
DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else "cpu")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)} (device 0 of {torch.cuda.device_count()})")
else:
    print(f"Using {DEVICE} (no CUDA GPU detected)")

def run(cmd: list[str], cwd: Path | None = None) -> int:
    cwd = cwd or ROOT
    print(" ".join(str(x) for x in cmd))
    return subprocess.run(cmd, cwd=cwd, env=env).returncode

GPU: NVIDIA RTX 6000 Ada Generation (device 0 of 3)


## 1. Preprocess

In [8]:
PREPROCESS_DIR = RUNS / "preprocess"
run([sys.executable, "-m", "bulkformer_dx.cli", "preprocess",
     "--counts", "data/clinical_rnaseq/raw_counts.tsv",
     "--annotation", "data/clinical_rnaseq/gene_annotation_v29.tsv",
     "--output-dir", str(PREPROCESS_DIR),
     "--counts-orientation", "genes-by-samples"])

/srv/miniconda3/envs/bulkformer-cuda/bin/python -m bulkformer_dx.cli preprocess --counts data/clinical_rnaseq/raw_counts.tsv --annotation data/clinical_rnaseq/gene_annotation_v29.tsv --output-dir /home/smirnov/projects/BulkFormer/runs/clinical_methods_37M/preprocess --counts-orientation genes-by-samples
Wrote preprocessing outputs to /home/smirnov/projects/BulkFormer/runs/clinical_methods_37M/preprocess
Samples: 146 | input genes: 60788 | BulkFormer-valid genes: 19751/20010


0

## 2. Anomaly score (37M)

In [9]:
SCORE_DIR = RUNS / "anomaly_score"
run([sys.executable, "-m", "bulkformer_dx.cli", "anomaly", "score",
     "--input", str(PREPROCESS_DIR / "aligned_log1p_tpm.tsv"),
     "--valid-gene-mask", str(PREPROCESS_DIR / "valid_gene_mask.tsv"),
     "--output-dir", str(SCORE_DIR),
     "--variant", "37M", "--device", DEVICE, "--mc-passes", "20", "--mask-prob", "0.10"])

/srv/miniconda3/envs/bulkformer-cuda/bin/python -m bulkformer_dx.cli anomaly score --input /home/smirnov/projects/BulkFormer/runs/clinical_methods_37M/preprocess/aligned_log1p_tpm.tsv --valid-gene-mask /home/smirnov/projects/BulkFormer/runs/clinical_methods_37M/preprocess/valid_gene_mask.tsv --output-dir /home/smirnov/projects/BulkFormer/runs/clinical_methods_37M/anomaly_score --variant 37M --device cuda --mc-passes 20 --mask-prob 0.10


/srv/miniconda3/envs/bulkformer-cuda/lib/python3.12/site-packages/torch_geometric/typing.py:68: UserWarning: An issue occurred while importing 'pyg-lib'. Disabling its usage. Stacktrace: Could not load this library: /srv/miniconda3/envs/bulkformer-cuda/lib/python3.12/site-packages/libpyg.so
  warnings.warn(f"An issue occurred while importing 'pyg-lib'. "
/srv/miniconda3/envs/bulkformer-cuda/lib/python3.12/site-packages/torch_geometric/typing.py:86: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: Could not load this library: /srv/miniconda3/envs/bulkformer-cuda/lib/python3.12/site-packages/torch_scatter/_version_cuda.so
  warnings.warn(f"An issue occurred while importing 'torch-scatter'. "
/srv/miniconda3/envs/bulkformer-cuda/lib/python3.12/site-packages/torch_geometric/typing.py:97: UserWarning: An issue occurred while importing 'torch-cluster'. Disabling its usage. Stacktrace: Could not load this library: /srv/miniconda3/envs/bulkformer

Wrote anomaly scoring outputs to /home/smirnov/projects/BulkFormer/runs/clinical_methods_37M/anomaly_score
Samples: 146 | valid genes: 19751 | MC passes: 20 | mean cohort abs residual: 0.8375


0

## 2b. Embeddings (for knn_local)

Extract sample embeddings for local-cohort calibration.

In [10]:
EMBEDDINGS_DIR = RUNS / "embeddings"
run([sys.executable, "-m", "bulkformer_dx.cli", "embeddings", "extract",
     "--input", str(PREPROCESS_DIR / "aligned_log1p_tpm.tsv"),
     "--valid-gene-mask", str(PREPROCESS_DIR / "valid_gene_mask.tsv"),
     "--output-dir", str(EMBEDDINGS_DIR), "--variant", "37M", "--device", DEVICE])

/srv/miniconda3/envs/bulkformer-cuda/bin/python -m bulkformer_dx.cli embeddings extract --input /home/smirnov/projects/BulkFormer/runs/clinical_methods_37M/preprocess/aligned_log1p_tpm.tsv --valid-gene-mask /home/smirnov/projects/BulkFormer/runs/clinical_methods_37M/preprocess/valid_gene_mask.tsv --output-dir /home/smirnov/projects/BulkFormer/runs/clinical_methods_37M/embeddings --variant 37M --device cuda


/srv/miniconda3/envs/bulkformer-cuda/lib/python3.12/site-packages/torch_geometric/typing.py:68: UserWarning: An issue occurred while importing 'pyg-lib'. Disabling its usage. Stacktrace: Could not load this library: /srv/miniconda3/envs/bulkformer-cuda/lib/python3.12/site-packages/libpyg.so
  warnings.warn(f"An issue occurred while importing 'pyg-lib'. "
/srv/miniconda3/envs/bulkformer-cuda/lib/python3.12/site-packages/torch_geometric/typing.py:86: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: Could not load this library: /srv/miniconda3/envs/bulkformer-cuda/lib/python3.12/site-packages/torch_scatter/_version_cuda.so
  warnings.warn(f"An issue occurred while importing 'torch-scatter'. "
/srv/miniconda3/envs/bulkformer-cuda/lib/python3.12/site-packages/torch_geometric/typing.py:97: UserWarning: An issue occurred while importing 'torch-cluster'. Disabling its usage. Stacktrace: Could not load this library: /srv/miniconda3/envs/bulkformer

Wrote /home/smirnov/projects/BulkFormer/runs/clinical_methods_37M/embeddings/sample_embeddings.tsv (146 samples x 131 dims)


0

## 3. Calibrate with all methods

In [11]:
# none (default)
run([sys.executable, "-m", "bulkformer_dx.cli", "anomaly", "calibrate",
     "--scores", str(SCORE_DIR),
     "--output-dir", str(RUNS / "calibrated_none"), "--alpha", "0.05"])

/srv/miniconda3/envs/bulkformer-cuda/bin/python -m bulkformer_dx.cli anomaly calibrate --scores /home/smirnov/projects/BulkFormer/runs/clinical_methods_37M/anomaly_score --output-dir /home/smirnov/projects/BulkFormer/runs/clinical_methods_37M/calibrated_none --alpha 0.05
Wrote calibrated anomaly outputs to /home/smirnov/projects/BulkFormer/runs/clinical_methods_37M/calibrated_none
Samples: 146 | scored genes: 2533004 | method: none
Outlier counts: mean=92.2 | median=46 | max=1848 | inflation=0.11x


0

In [ ]:
# nb_approx (TPM-derived NB approximation)
run([sys.executable, "-m", "bulkformer_dx.cli", "anomaly", "calibrate",
     "--scores", str(SCORE_DIR),
     "--output-dir", str(RUNS / "calibrated_nb_approx"),
     "--count-space-method", "nb_approx", "--alpha", "0.05"])

/srv/miniconda3/envs/bulkformer-cuda/bin/python -m bulkformer_dx.cli anomaly calibrate --scores /home/smirnov/projects/BulkFormer/runs/clinical_methods_37M/anomaly_score --output-dir /home/smirnov/projects/BulkFormer/runs/clinical_methods_37M/calibrated_nb_approx --count-space-method nb_approx --alpha 0.05


In [ ]:
# nb_outrider (OUTRIDER-style count-space NB)
run([sys.executable, "-m", "bulkformer_dx.cli", "anomaly", "calibrate",
     "--scores", str(SCORE_DIR),
     "--output-dir", str(RUNS / "calibrated_nb_outrider"),
     "--count-space-method", "nb_outrider",
     "--count-space-path", str(PREPROCESS_DIR), "--alpha", "0.05"])

In [ ]:
# student_t (heavy-tailed; recommended for omics residuals)
run([sys.executable, "-m", "bulkformer_dx.cli", "anomaly", "calibrate",
     "--scores", str(SCORE_DIR),
     "--output-dir", str(RUNS / "calibrated_student_t"),
     "--student-t", "--alpha", "0.05"])

In [ ]:
# Create embeddings.npy for knn_local (calibration expects .npy, embeddings extract writes .tsv)
import numpy as np
import pandas as pd
emb_tsv = EMBEDDINGS_DIR / "sample_embeddings.tsv"
cohort_tsv = SCORE_DIR / "cohort_scores.tsv"
emb_npy = SCORE_DIR / "embeddings.npy"
if emb_tsv.exists() and cohort_tsv.exists():
    cohort = pd.read_csv(cohort_tsv, sep="\t")
    sample_ids = cohort["sample_id"].astype(str).tolist()
    emb_df = pd.read_csv(emb_tsv, sep="\t")
    emb_df = emb_df.set_index("sample_id").loc[sample_ids]
    np.save(emb_npy, emb_df.values.astype(np.float32))
    print(f"Saved {emb_npy} for knn_local")

In [ ]:
# knn_local (local cohort: kNN in embedding space)
run([sys.executable, "-m", "bulkformer_dx.cli", "anomaly", "calibrate",
     "--scores", str(SCORE_DIR),
     "--output-dir", str(RUNS / "calibrated_knn_local"),
     "--cohort-mode", "knn_local", "--knn-k", "50",
     "--embedding-path", str(SCORE_DIR / "embeddings.npy"), "--alpha", "0.05"])

## 2c. NLL scoring (TabPFN-style pseudo-likelihood)

Different scoring method: accumulates log-probabilities for masked genes instead of residuals.

In [ ]:
SCORE_NLL_DIR = RUNS / "anomaly_score_nll"
run([sys.executable, "-m", "bulkformer_dx.cli", "anomaly", "score",
     "--input", str(PREPROCESS_DIR / "aligned_log1p_tpm.tsv"),
     "--valid-gene-mask", str(PREPROCESS_DIR / "valid_gene_mask.tsv"),
     "--output-dir", str(SCORE_NLL_DIR),
     "--variant", "37M", "--device", DEVICE, "--mc-passes", "8", "--mask-prob", "0.15",
     "--score-type", "nll"])

In [ ]:
# NLL + calibrate (none)
run([sys.executable, "-m", "bulkformer_dx.cli", "anomaly", "calibrate",
     "--scores", str(SCORE_NLL_DIR),
     "--output-dir", str(RUNS / "calibrated_nll"), "--alpha", "0.05"])

## 4. Load calibration outputs

In [ ]:
import json
import pandas as pd

METHODS = ["none", "student_t", "nb_approx", "nb_outrider", "knn_local", "nll"]
summaries = {}
outliers = {}
ranked = {}

for m in METHODS:
    base = RUNS / f"calibrated_{m}"
    if (base / "calibration_summary.tsv").exists():
        summaries[m] = pd.read_csv(base / "calibration_summary.tsv", sep="\t")
    if (base / "absolute_outliers.tsv").exists():
        outliers[m] = pd.read_csv(base / "absolute_outliers.tsv", sep="\t")
    rdir = base / "ranked_genes"
    if rdir.exists():
        ranked[m] = {p.stem: pd.read_csv(p, sep="\t") for p in sorted(rdir.glob("*.tsv"))}

print("Loaded:", list(summaries.keys()))

## 5. Outlier counts per sample (by method)

In [ ]:
import matplotlib.pyplot as plt

FIGURES.mkdir(parents=True, exist_ok=True)

if summaries:
    fig, ax = plt.subplots(figsize=(10, 4))
    for m in METHODS:
        if m in summaries and "absolute_significant_gene_count_by_alpha" in summaries[m].columns:
            s = summaries[m].sort_values("sample_id")
            ax.hist(s["absolute_significant_gene_count_by_alpha"], bins=25, alpha=0.6, label=m)
    ax.set_xlabel("Outliers per sample")
    ax.set_ylabel("Count")
    ax.set_title("Outlier count distribution by calibration method")
    ax.legend()
    fig.tight_layout()
    fig.savefig(FIGURES / "outliers_per_sample_by_method.png", dpi=150)
    plt.show()
    print(f"Saved {FIGURES / 'outliers_per_sample_by_method.png'}")

## 6. P-value distributions

In [ ]:
import numpy as np
if ranked:
    n_methods = len([m for m in METHODS if m in ranked and ranked[m]])
    n_cols = min(3, n_methods)
    n_rows = (n_methods + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows))
    axes = np.atleast_2d(axes)
    if n_methods == 1:
        axes = axes.reshape(1, -1)
    idx = 0
    for m in METHODS:
        if m not in ranked or not ranked[m]:
            continue
        ax = axes.flat[idx]
        tbl = next(iter(ranked[m].values()))
        if m == "nb_outrider" and "nb_outrider_p_raw" in tbl.columns:
            p = tbl["nb_outrider_p_raw"].dropna()
            ax.hist(p, bins=30, alpha=0.7, color="coral")
            ax.set_xlabel("NB Outrider p-value")
        elif m == "nb_approx" and "nb_approx_two_sided_p_value" in tbl.columns:
            p = tbl["nb_approx_two_sided_p_value"].dropna()
            ax.hist(p, bins=30, alpha=0.7, color="seagreen")
            ax.set_xlabel("NB approx p-value")
        elif m == "student_t" and "raw_p_value" in tbl.columns:
            p = tbl["raw_p_value"].dropna()
            ax.hist(p, bins=30, alpha=0.7, color="darkorange")
            ax.set_xlabel("Student-t p-value")
        elif "raw_p_value" in tbl.columns:
            p = tbl["raw_p_value"].dropna()
            ax.hist(p, bins=30, alpha=0.7, color="steelblue")
            ax.set_xlabel("Gaussian p-value")
        elif "empirical_p_value" in tbl.columns:
            p = tbl["empirical_p_value"].dropna()
            ax.hist(p, bins=30, alpha=0.7, color="gray")
            ax.set_xlabel("Empirical p-value")
        else:
            continue
        ax.set_ylabel("Count")
        ax.set_title(m)
        idx += 1
    for j in range(idx, axes.size):
        axes.flat[j].set_visible(False)
    fig.tight_layout()
    fig.savefig(FIGURES / "pvalue_distributions.png", dpi=150)
    plt.show()
    print(f"Saved {FIGURES / 'pvalue_distributions.png'}")

## 7. Method comparison summary table

In [ ]:
rows = []
for m in METHODS:
    if m not in summaries:
        continue
    s = summaries[m]
    col = "absolute_significant_gene_count_by_alpha"
    if col in s.columns:
        rows.append({
            "method": m,
            "mean_outliers": s[col].mean(),
            "median_outliers": s[col].median(),
            "max_outliers": s[col].max(),
        })
if rows:
    df = pd.DataFrame(rows)
    display(df)

## 8. Gene overlap (significant genes across methods)

In [ ]:
def get_sig_genes(outliers_df: pd.DataFrame) -> set[tuple[str, str]]:
    sig = outliers_df[outliers_df["is_significant"]][["sample_id", "gene"]].drop_duplicates()
    return set(zip(sig["sample_id"].astype(str), sig["gene"].astype(str)))

if outliers:
    sig_sets = {m: get_sig_genes(outliers[m]) for m in METHODS if m in outliers}
    if len(sig_sets) >= 2:
        methods = list(sig_sets.keys())
        venn_ok = False
        if len(methods) <= 3:
            try:
                from matplotlib_venn import venn2, venn3
                if len(methods) == 2:
                    fig, ax = plt.subplots(figsize=(6, 6))
                    venn2([sig_sets[methods[0]], sig_sets[methods[1]]], set_labels=methods)
                else:
                    fig, ax = plt.subplots(figsize=(8, 8))
                    venn3([sig_sets[methods[0]], sig_sets[methods[1]], sig_sets[methods[2]]], set_labels=methods)
                ax.set_title("Overlap of significant (sample, gene) calls")
                fig.savefig(FIGURES / "method_overlap_venn.png", dpi=150)
                plt.show()
                print(f"Saved {FIGURES / 'method_overlap_venn.png'}")
                venn_ok = True
            except ImportError:
                pass
        if not venn_ok or len(methods) > 3:
            fig, axes = plt.subplots(1, 2, figsize=(14, 5))
            colors = ["steelblue", "coral", "seagreen", "darkorange", "purple", "teal"][:len(methods)]
            axes[0].bar(methods, [len(sig_sets[m]) for m in methods], color=colors)
            axes[0].set_ylabel("Total significant (sample, gene) calls")
            axes[0].set_title("Outlier calls per method")
            axes[0].tick_params(axis="x", rotation=45)
            labels, vals = [], []
            for i, m1 in enumerate(methods):
                for m2 in methods[i+1:]:
                    labels.append(f"{m1}∩{m2}")
                    vals.append(len(sig_sets[m1] & sig_sets[m2]))
            axes[1].bar(range(len(vals)), vals)
            axes[1].set_xticks(range(len(labels)))
            axes[1].set_xticklabels(labels, rotation=45, ha="right")
            axes[1].set_ylabel("Overlap")
            axes[1].set_title("Pairwise overlap")
            fig.tight_layout()
            fig.savefig(FIGURES / "method_overlap.png", dpi=150)
            plt.show()
            print(f"Saved {FIGURES / 'method_overlap.png'}")
    else:
        print("Need at least 2 methods for overlap plot.")
else:
    print("No outlier tables loaded.")

In [ ]:
# Fallback: overlap table if matplotlib_venn not installed
def overlap_table(sig_sets: dict[str, set]) -> pd.DataFrame:
    methods = list(sig_sets.keys())
    rows = []
    for m in methods:
        for m2 in methods:
            if m <= m2:
                inter = len(sig_sets[m] & sig_sets[m2])
                rows.append({"method_a": m, "method_b": m2, "overlap": inter})
    return pd.DataFrame(rows)

if outliers and len(outliers) >= 2:
    sig_sets = {m: get_sig_genes(outliers[m]) for m in METHODS if m in outliers}
    display(overlap_table(sig_sets))

## 9. Z-score distribution (absolute approach)

In [ ]:
if "none" in outliers:
    o = outliers["none"]
    fig, ax = plt.subplots(figsize=(8, 4))
    sig = o["is_significant"]
    ax.hist(o.loc[~sig, "z_score"], bins=80, alpha=0.7, color="steelblue", label="non-outlier")
    if sig.any():
        ax.hist(o.loc[sig, "z_score"], bins=50, alpha=0.8, color="coral", label="outlier")
    ax.axvline(-2, color="orange", linestyle="--", label="z=±2")
    ax.axvline(2, color="orange", linestyle="--")
    ax.set_xlabel("z-score")
    ax.set_ylabel("Count")
    ax.set_title("Z-score distribution (method=none)")
    ax.legend()
    fig.tight_layout()
    fig.savefig(FIGURES / "zscore_distribution.png", dpi=150)
    plt.show()
    print(f"Saved {FIGURES / 'zscore_distribution.png'}")